# Study report: isr-to-thread

The study measures the ISR to thread wake latency (M2): the time from
the ISR to the first instruction of the thread that the ISR wakes. It
uses the app `wake_latency` and the five kernel objects that an ISR can
use to wake a thread. It also asks which setting makes the two modes of
the latency.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-default-latencies@nucleo_g474re/st-isr-to-thread"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

The study has no `hypotheses.yaml`. The experiment files give no
prediction.

## The modes of a histogram

A mode is a group of bins that each hold at least 0.1 % of the samples,
with at most 2 cycles between neighbours. Mode 1 is the fastest.
`mode_share(k)` of the checks is the share of the samples in mode k
(`hypotheses.yaml`, `scripts/benchmarks/studylib/stats.py`). The mode
tables use the same rule (`nbfig.modes`), so they show what the checks
see. This rule reads the fine bins only. Where a table has the columns
"... with coarse bins", these columns use the same rule over the fine
and the coarse bins: a coarse bin of 32 cycles is one bin. A difference
between the two views is a mode that the checks do not see.

The app data keep only the bins with samples. In the histograms, the
line goes down to a floor at each end of a group of bins, and an empty
bin is a gap.

## Kernel objects and ISR kinds (wake-isr-1m)

The woken thread has priority 0. It is preemptible, and time slicing can
apply to it. The kernel objects (app page of `wake_latency`): `sem`
(`k_sem_give()`), `event` (`k_event_post()`), `msgq` (`k_msgq_put()`,
4-byte message), `fifo` (`k_fifo_put()`) and `poll`
(`k_poll_signal_raise()`). The ISR kinds: `wrapped` runs through
`_isr_wrapper` (the default), `direct` is a direct ISR
(`LATBENCH_WAKE_ISR_DIRECT`, point `direct-sem` only). The central result
comes from the point `wrapped-sem`.

The menu selects the metric: `isr2thr` is M2, `irq` is the edge to the
ISR, `irq2thr` is the edge to the woken thread.

In [ ]:
df = inst.rows("wake-isr-1m")
nbfig.dots(df, "ISR to thread per kernel object and ISR kind (wake-isr-1m)",
           stats=("p50", "p99", "p99.99", "max")).show()
nbfig.pivot(df, ["p50", "p99", "p99.99", "max"])

In [ ]:
nbfig.ccdf(nbfig.hists_of(inst, "wake-isr-1m"), "wake-isr-1m: CCDF").show()

## The two modes (wake-isr-1m)

The table gives the modes of `isr2thr` for each point. The column
`step` is the distance between the first values of the two largest
modes. The histogram shows the bins near the modes.

In [ ]:
def with_step(t, experiment, metric):
    """Add the column step: the distance between the first values of the two largest modes."""
    t = t.copy()
    step = {}
    for run, tree, point, rep in inst.points(experiment):
        m = sorted(nbfig.modes(rep["hists"][metric]), key=lambda x: -x[2])[:2]
        step[(experiment, point)] = abs(m[1][0] - m[0][0]) if len(m) == 2 else None
    t["step"] = [step.get(i) for i in t.index]
    return t


with_step(nbfig.mode_table(inst, ["wake-isr-1m"], "isr2thr"), "wake-isr-1m", "isr2thr")

In [ ]:
series = nbfig.hists_of(inst, "wake-isr-1m")
nbfig.hist(series, "ISR to thread near the modes (wake-isr-1m)", ["isr2thr"],
           xrange=nbfig.mode_range(series, "isr2thr", 40)).show()

## The setting that makes the two modes (wake-anchors-1m)

The study `zephyr-versions` owns `wake-anchors-1m`. This study does not
import it, because no check of this study uses it. The notebook shows
it, if the results tree has it. The points are the cube `anchor` x
`arch`:

- `default`: the board and kernel defaults, the reference.
- `noslice`: only time slicing off (`TIMESLICING: n`).
- `lean`: `common/conf/lean.conf` (no time slicing, no stack guard,
  `-O2`).
- `legacy` is the thread switch through PendSV, `switch` is the new arch
  interface (`common/conf/use_switch.conf`).

In [ ]:
if nbfig.built(inst, "wake-anchors-1m"):
    series = nbfig.hists_of(inst, "wake-anchors-1m")
    nbfig.hist(series, "ISR to thread near the modes per anchor (wake-anchors-1m)", ["isr2thr"],
               xrange=nbfig.mode_range(series, "isr2thr", 40)).show()
    display(with_step(nbfig.mode_table(inst, ["wake-anchors-1m"], "isr2thr"), "wake-anchors-1m", "isr2thr"))

## The fast mode against the stimulus (wake-fast-mode)

Does the fast mode (mode 1) depend on the stimulus delay, or on other
state? Each point changes one factor against `r100-500` (the app
defaults, semaphore from the wrapped ISR):

- `r100-200` and `r400-500`: other ranges of the random stimulus
  interval.
- `fixed300`: a fixed period of 300 us, plus 1 cycle per edge.
- `poll1000`: the poll interval of the main thread is 1000 ms, not
  100 ms.

The design gives two readings. If the share of the fast mode follows the
stimulus (for example 0 % or 100 % with a fixed period), the cause
depends on the delay. If the share stays as at `r100-500`, the cause is
other state. The dashed line is the share at `r100-500`. The study has
no `hypotheses.yaml`.

In [ ]:
import plotly.graph_objects as go

rows = []
for run, tree, point, rep in inst.points("wake-fast-mode"):
    m = nbfig.modes(rep["hists"]["isr2thr"])
    rows.append({"point": point, "modes": len(m), "mode 1": m[0][0], "share of mode 1": round(m[0][2], 4),
                 "mode 2": m[1][0] if len(m) > 1 else None, "share of mode 2": round(m[1][2], 4) if len(m) > 1 else 0})
fast = pd.DataFrame(rows)
fig = go.Figure(go.Bar(x=fast.point, y=fast["share of mode 1"], name="share of mode 1", marker_color=nbfig.color(0),
                       hovertemplate="%{x}: %{y:.4f}<extra>mode 1</extra>"))
ref = fast.loc[fast.point == "r100-500", "share of mode 1"]
if len(ref):
    fig.add_hline(y=ref.iloc[0], line_dash="dash", line_color="#6e6d66", annotation_text="r100-500",
                  annotation_position="top right")
nbfig.style(fig, "Share of the fast mode of isr2thr (wake-fast-mode)", "point", "share of the samples",
            yaxis_range=[0, 1]).show()
fast.set_index("point")

In [ ]:
df = inst.rows("wake-fast-mode")
nbfig.pivot(df, ["p50", "p99", "p99.99", "max", "mean"])